# AST-GR Slate Policy — Kaggle T4×2 pilot

Notebook tự chứa pipeline pilot cho repository-level code completion: AST chunking → proposal pool BM25 + frozen AlignRetriever → typed AST/relation slate policy → scalar reward từ frozen code generator → generation/evaluation.

## Model nào làm gì

- Frozen retriever/encoder: AlignCoder/AlignRetriever. Không update encoder trong pilot.
- Model được train: một SlatePolicy nhỏ (action head + value baseline), không phải generator. Train bằng torchrun trên 2 GPU.
- Frozen reward evaluator và final code generator: deepseek-ai/deepseek-coder-1.3b-base, chạy một lần trong vLLM server nền.
- Loss chính: REINFORCE trên đúng slate đã chọn, reward là delta average log P(gold continuation | prompt) giữa có retrieval và không retrieval. Value MSE chỉ giảm variance.
- Không distillation, không tạo preference pairs/ranking loss, không LiPO. Ma trận cạnh AST là structural feature xác định từ parser + identifier heuristic, không phải dữ liệu pairwise để train.

## Trước khi chạy

1. Chọn GPU T4×2 trong Kaggle Settings, bật Internet để tải model/runtime, hoặc gắn sẵn model và dataset vào Input.
2. Gắn Data4AlignCoder (có github_repos train parquet). Nếu chỉ gắn benchmark, notebook sẽ thử tải Data4AlignCoder công khai.
3. Chạy tuần tự từ trên xuống. RUN_MODE mặc định là smoke để kiểm tra đường đi end-to-end; đổi sang full sau khi smoke hoàn tất.

vLLM chạy trên GPU 0 với giới hạn VRAM; torchrun dùng 2 process. Trong pha RL, encoder đã được giải phóng nên chỉ policy head nhỏ chia GPU 0 với server. Nếu server hoặc NCCL báo OOM, giảm VLLM_GPU_MEMORY_UTILIZATION hoặc đặt NPROC=1.

Đầu ra nằm trong /kaggle/working/ast_gr_slate. Baseline là dense top-budget trên cùng candidate pool AST/BM25/dense; đây không phải reproduction đầy đủ AlignCoder, nên kết quả không được diễn giải là đã vượt AlignCoder nếu chưa chạy baseline chính thức trên cùng generator, dữ liệu và ngân sách.

In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import random
import shutil
import subprocess
import importlib.util

RUN_MODE = "smoke"  # "smoke" or "full"
SEED = 17
DOWNLOAD_DATA_IF_MISSING = True
DATASET_REPO = "AlignCoder/Data4AlignCoder"
DATASET_INPUT_HINT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/ast_gr_slate")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("/kaggle/working/hf-cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

RETRIEVER_MODEL = "AlignCoder/AlignRetriever"
GENERATOR_MODEL = "deepseek-ai/deepseek-coder-1.3b-base"
SERVED_MODEL_NAME = "deepseek-coder-1.3b-base"
API_BASE = "http://127.0.0.1:8000/v1"

# Small defaults for an end-to-end smoke test; the full setting is still Kaggle-bounded.
if RUN_MODE == "smoke":
    TRAIN_EXAMPLES = 24
    VALID_EXAMPLES = 8
    MAX_REPO_GROUPS = 240
    EPOCHS = 2
    EVAL_LIMIT = 16
else:
    TRAIN_EXAMPLES = 400
    VALID_EXAMPLES = 64
    MAX_REPO_GROUPS = 1800
    EPOCHS = 3
    EVAL_LIMIT = -1

MAX_FILES_PER_REPO = 24
MAX_RELATED_FILES = 10
MAX_CHUNKS_PER_FILE = 24
AST_CHUNK_TOKENS = 192
TARGET_CHUNK_TOKENS = 96
MIN_TARGET_TOKENS = 8
MIN_PREFIX_TOKENS = 32

RETRIEVER_QUERY_LENGTH = 256
RETRIEVER_CANDIDATE_LENGTH = 256
RETRIEVER_BATCH_SIZE = 24
BM25_PROPOSALS = 12
DENSE_PROPOSALS = 12
CANDIDATE_POOL_SIZE = 16
CROSSFILE_TOKEN_BUDGET = 384
MAX_SLATE_STEPS = 3
GENERATOR_MAX_MODEL_LEN = 2048
GENERATOR_TARGET_TOKENS = 96
GENERATOR_OUTPUT_TOKENS = 128

VLLM_GPU_MEMORY_UTILIZATION = 0.40
VLLM_MAX_NUM_SEQS = 4
VLLM_PORT = 8000
NPROC = 2

print("Run mode:", RUN_MODE)
print("Artifacts:", WORK_DIR)


In [ ]:
# Only install small parsing/retrieval dependencies if Kaggle image lacks them.
LIGHT_PACKAGES = [
    ("tree-sitter", "tree_sitter"),
    ("tree-sitter-python", "tree_sitter_python"),
    ("tree-sitter-java", "tree_sitter_java"),
    ("rank-bm25", "rank_bm25"),
]
missing = [package for package, module in LIGHT_PACKAGES
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Visible GPU count:", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_idx)
    print(f"GPU {gpu_idx}: {props.name}; {props.total_memory / 1024**3:.1f} GiB")

if torch.cuda.device_count() < 2:
    raise RuntimeError("Notebook này được cấu hình cho T4×2 và torchrun 2 process. Hãy chọn Accelerator = GPU T4×2.")


In [ ]:
def locate_data_root():
    mounts = []
    if DATASET_INPUT_HINT.exists():
        mounts.extend(sorted(p for p in DATASET_INPUT_HINT.iterdir() if p.is_dir()))
    mounts.extend([Path("/kaggle/working"), Path.cwd()])

    candidates = []
    for mount in mounts:
        candidates.extend([mount, mount / "data", mount / "Data4AlignCoder",
                           mount / "Data4AlignCoder" / "data"])
    for candidate in candidates:
        if (candidate / "github_repos" / "python" / "train.parquet").is_file():
            return candidate.resolve()

    if not DOWNLOAD_DATA_IF_MISSING:
        raise FileNotFoundError(
            "Không thấy github_repos/python/train.parquet. Hãy gắn Data4AlignCoder "
            "vào /kaggle/input hoặc bật DOWNLOAD_DATA_IF_MISSING."
        )

    from huggingface_hub import snapshot_download
    destination = Path("/kaggle/working/data4aligncoder")
    if not (destination / "github_repos" / "python" / "train.parquet").is_file():
        print("Tải Data4AlignCoder từ Hugging Face; dataset công khai khoảng 2.18 GB.")
        snapshot_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            local_dir=str(destination),
            cache_dir=str(CACHE_DIR / "hub"),
        )
    if (destination / "data" / "github_repos" / "python" / "train.parquet").is_file():
        return (destination / "data").resolve()
    if (destination / "github_repos" / "python" / "train.parquet").is_file():
        return destination.resolve()
    raise FileNotFoundError(
        "Đã tải snapshot nhưng không tìm được github_repos/python/train.parquet. "
        "Kiểm tra lại cấu trúc dataset mount."
    )

DATA_ROOT = locate_data_root()
print("Dataset root:", DATA_ROOT)
print("Train parquet:", DATA_ROOT / "github_repos/python/train.parquet")


In [ ]:
import re
import hashlib
from dataclasses import dataclass
from collections import defaultdict

import tree_sitter as ts
import tree_sitter_python as ts_python
import tree_sitter_java as ts_java
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GEN_TOKENIZER = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL, cache_dir=str(CACHE_DIR), use_fast=True
)
if GEN_TOKENIZER.pad_token_id is None:
    GEN_TOKENIZER.pad_token = GEN_TOKENIZER.eos_token

SEMANTIC_NODE_TYPES = {
    "function_definition", "class_definition",
    "method_declaration", "constructor_declaration",
    "class_declaration", "interface_declaration", "enum_declaration",
}
IDENTIFIER_RE = re.compile(r"[A-Za-z_$][A-Za-z0-9_$]*")
DEF_RE = re.compile(
    r"\b(?:def|class|interface|enum|struct|record)\s+([A-Za-z_$][A-Za-z0-9_$]*)"
)
COMMON_WORDS = {
    "def", "class", "return", "if", "else", "elif", "for", "while", "try",
    "except", "finally", "with", "as", "import", "from", "package", "public",
    "private", "protected", "static", "void", "int", "long", "boolean", "new",
    "this", "self", "true", "false", "null", "none", "string", "return",
}

@dataclass
class ASTUnit:
    file_path: str
    language: str
    text: str
    start_byte: int
    end_byte: int
    node_type: str
    scope_key: str
    definitions: set
    references: set
    token_cost: int = 0
    is_parse_error: bool = False

def _language_object(language):
    language = language.lower()
    if language in ("py", "python"):
        return ts.Language(ts_python.language())
    if language in ("java",):
        return ts.Language(ts_java.language())
    raise ValueError(f"AST parser not configured for language={language!r}")

def _node_name(node, source_bytes):
    for child in node.children:
        if child.type in {"identifier", "name", "type_identifier", "field_identifier"}:
            value = source_bytes[child.start_byte:child.end_byte].decode("utf-8", errors="replace")
            if value.strip():
                return value.strip()
    first = source_bytes[node.start_byte:min(node.end_byte, node.start_byte + 160)]
    return first.decode("utf-8", errors="replace").splitlines()[0].strip()[:80]

def _symbol_sets(code):
    lowered = set(IDENTIFIER_RE.findall(code))
    definitions = set(DEF_RE.findall(code))
    references = lowered - definitions - COMMON_WORDS
    return definitions, references

def ast_chunk_file(file_path, code, language, tokenizer, max_tokens, max_chunks=MAX_CHUNKS_PER_FILE):
    source = code.encode("utf-8", errors="replace")
    parser = ts.Parser(_language_object(language))
    tree = parser.parse(source)
    units = []

    def count_tokens(raw_bytes):
        text = raw_bytes.decode("utf-8", errors="replace")
        if not text.strip():
            return 0
        return len(tokenizer.encode(text, add_special_tokens=False))

    def emit(start_byte, end_byte, node_type, scope_key, parse_error):
        if end_byte <= start_byte:
            return
        text = source[start_byte:end_byte].decode("utf-8", errors="replace")
        if not text.strip():
            return
        token_cost = count_tokens(source[start_byte:end_byte])
        if token_cost == 0:
            return
        defs, refs = _symbol_sets(text)
        units.append(ASTUnit(
            file_path=str(file_path), language=language, text=text,
            start_byte=int(start_byte), end_byte=int(end_byte),
            node_type=str(node_type), scope_key=str(scope_key),
            definitions=defs, references=refs, token_cost=token_cost,
            is_parse_error=bool(parse_error),
        ))

    def visit(node, ancestor_scope):
        if len(units) >= max_chunks:
            return
        raw = source[node.start_byte:node.end_byte]
        n_tokens = count_tokens(raw)
        own_scope = ancestor_scope
        if node.type in SEMANTIC_NODE_TYPES:
            name = _node_name(node, source)
            own_scope = ancestor_scope + [f"{node.type}:{name}"]
        if n_tokens and n_tokens <= max_tokens:
            emit(node.start_byte, node.end_byte, node.type, "/".join(own_scope), tree.root_node.has_error)
            return

        children = [child for child in node.children if child.end_byte > child.start_byte]
        if not children:
            # Extremely large leaf literals are omitted rather than split on physical lines.
            return

        pending_start = None
        pending_end = None

        def flush_pending():
            nonlocal pending_start, pending_end
            if pending_start is not None and pending_end is not None:
                emit(pending_start, pending_end, f"{node.type}:ast_window",
                     "/".join(own_scope), tree.root_node.has_error)
            pending_start, pending_end = None, None

        for child in children:
            if len(units) >= max_chunks:
                break
            child_raw = source[child.start_byte:child.end_byte]
            child_tokens = count_tokens(child_raw)
            if child_tokens > max_tokens:
                flush_pending()
                visit(child, own_scope)
                continue

            proposed_start = pending_start if pending_start is not None else node.start_byte
            if pending_start is None and count_tokens(
                source[proposed_start:child.end_byte]
            ) > max_tokens:
                # A preceding oversized AST child may already have been recursed into;
                # start a fresh sibling window here instead of re-including that subtree.
                proposed_start = child.start_byte
            proposed_end = child.end_byte
            proposed_tokens = count_tokens(source[proposed_start:proposed_end])
            if pending_start is not None and proposed_tokens > max_tokens:
                flush_pending()
                pending_start, pending_end = child.start_byte, child.end_byte
            else:
                if pending_start is None:
                    pending_start = proposed_start
                pending_end = child.end_byte
        flush_pending()

    visit(tree.root_node, [])
    # Stable source-span deduplication; parser errors remain observable in metadata.
    unique = {}
    for unit in units:
        unique[(unit.start_byte, unit.end_byte, unit.node_type)] = unit
    result = sorted(unique.values(), key=lambda u: (u.start_byte, u.end_byte))
    return result[:max_chunks], bool(tree.root_node.has_error)

def read_repository_groups(data_root, language, max_groups):
    parquet_path = data_root / "github_repos" / language / "train.parquet"
    if not parquet_path.is_file():
        print(f"Skip missing training parquet: {parquet_path}")
        return []
    parquet_file = pq.ParquetFile(parquet_path)
    groups, current = [], []
    max_files = MAX_FILES_PER_REPO

    for batch in parquet_file.iter_batches(
        batch_size=512, columns=["path", "content", "first"]
    ):
        for row in batch.to_pylist():
            if row.get("first") and current:
                if len(current) > 1:
                    groups.append((current, language))
                current = []
                if len(groups) >= max_groups:
                    break
            content = row.get("content")
            path = row.get("path")
            if path and content and len(current) < max_files:
                current.append((str(path), str(content)))
        if len(groups) >= max_groups:
            break
    if len(groups) < max_groups and len(current) > 1:
        groups.append((current, language))
    return groups[:max_groups]

def load_repository_splits(data_root):
    per_language = max(2, MAX_REPO_GROUPS // 2)
    groups = []
    for language in ("python", "java"):
        groups.extend(read_repository_groups(data_root, language, per_language))
    rng = random.Random(SEED)
    rng.shuffle(groups)
    n_valid = max(1, int(0.15 * len(groups)))
    return groups[n_valid:], groups[:n_valid]

def build_synthetic_examples(groups, count, split_name, seed):
    rng = random.Random(seed)
    examples = []
    if not groups:
        return examples
    max_attempts = max(500, count * 60)
    for attempt in range(max_attempts):
        if len(examples) >= count:
            break
        files, language = groups[rng.randrange(len(groups))]
        if len(files) < 2:
            continue
        target_index = rng.randrange(len(files))
        target_path, target_source = files[target_index]
        units, _ = ast_chunk_file(
            target_path, target_source, language, GEN_TOKENIZER,
            TARGET_CHUNK_TOKENS, max_chunks=48,
        )
        eligible = []
        for unit in units:
            if not (MIN_TARGET_TOKENS <= unit.token_cost <= TARGET_CHUNK_TOKENS):
                continue
            prefix = target_source.encode("utf-8", errors="replace")[:unit.start_byte].decode(
                "utf-8", errors="replace"
            )
            prefix_count = len(GEN_TOKENIZER.encode(prefix, add_special_tokens=False))
            if prefix_count >= MIN_PREFIX_TOKENS and unit.text.strip():
                eligible.append((unit, prefix))
        if not eligible:
            continue
        target_unit, prefix = rng.choice(eligible)
        related = [(path, text) for i, (path, text) in enumerate(files) if i != target_index]
        if len(related) > MAX_RELATED_FILES:
            related = rng.sample(related, MAX_RELATED_FILES)
        examples.append({
            "task_id": f"{split_name}-{len(examples)}",
            "file_path": target_path,
            "left_context": prefix,
            "target_code": target_unit.text,
            "language": language,
            "related_files": related,
        })
    if len(examples) < count:
        print(f"Note: created {len(examples)}/{count} {split_name} examples after filtering.")
    return examples

train_groups, valid_groups = load_repository_splits(DATA_ROOT)
print("Repository groups:", len(train_groups), "train /", len(valid_groups), "validation")
train_examples = build_synthetic_examples(train_groups, TRAIN_EXAMPLES, "train", SEED + 1)
valid_examples = build_synthetic_examples(valid_groups, VALID_EXAMPLES, "valid", SEED + 2)
if len(train_examples) < 2:
    raise RuntimeError(
        "Không tạo đủ synthetic training examples từ github_repos. "
        "Kiểm tra parquet có path/content/first và Python/Java source."
    )
print("Synthetic examples:", len(train_examples), "train /", len(valid_examples), "validation")


In [ ]:
def aligncoder_token_ids(text, tokenizer, max_length, is_query):
    tokens = tokenizer.tokenize(text)
    room = max_length - 4
    tokens = tokens[-room:] if is_query else tokens[:room]
    sequence = [tokenizer.cls_token, "<encoder-only>", tokenizer.sep_token]
    sequence += tokens + [tokenizer.sep_token]
    ids = tokenizer.convert_tokens_to_ids(sequence)
    ids = ids[:max_length]
    return ids + [tokenizer.pad_token_id] * (max_length - len(ids))

@torch.inference_mode()
def encode_retriever_texts(model, tokenizer, texts, max_length, is_query, device):
    if not texts:
        return torch.empty((0, model.config.hidden_size), dtype=torch.float32)
    embeddings = []
    for start in range(0, len(texts), RETRIEVER_BATCH_SIZE):
        batch_texts = texts[start:start + RETRIEVER_BATCH_SIZE]
        ids = [aligncoder_token_ids(t, tokenizer, max_length, is_query) for t in batch_texts]
        input_ids = torch.tensor(ids, dtype=torch.long, device=device)
        attention_mask = input_ids.ne(tokenizer.pad_token_id)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        weights = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1)
        embeddings.append(torch.nn.functional.normalize(pooled.float(), p=2, dim=-1).cpu())
    return torch.cat(embeddings, dim=0)

def render_unit(unit):
    marker = "#" if unit.language.lower() in ("python", "py") else "//"
    description = [
        f"file path: {unit.file_path}",
        f"AST node: {unit.node_type}",
        f"scope: {unit.scope_key or '<module>'}",
        f"span bytes: {unit.start_byte}-{unit.end_byte}",
    ]
    header = "\n".join(f"{marker} {line}" for line in description)
    body = "\n".join(f"{marker} {line}" for line in unit.text.splitlines())
    return (header + "\n" + body).strip()

def typed_relation_matrix(units):
    # 0 none, 1 nested AST scope, 2 same AST scope/siblings,
    # 3 identifier def-use heuristic, 4 shared-identifier heuristic.
    n = len(units)
    edges = np.zeros((n, n), dtype=np.int64)
    for i in range(n):
        for j in range(i + 1, n):
            left, right = units[i], units[j]
            relation = 0
            if left.file_path == right.file_path:
                if left.scope_key == right.scope_key:
                    relation = 2
                elif (left.scope_key and right.scope_key and
                      (left.scope_key.startswith(right.scope_key + "/") or
                       right.scope_key.startswith(left.scope_key + "/"))):
                    relation = 1
                elif left.node_type.endswith("ast_window") and right.node_type.endswith("ast_window"):
                    relation = 2
            if not relation:
                if (left.definitions & right.references) or (right.definitions & left.references):
                    relation = 3
                elif len(left.references & right.references) >= 2:
                    relation = 4
            edges[i, j] = relation
            edges[j, i] = relation
    return edges

def lexical_tokens(text):
    return re.findall(r"[A-Za-z_$][A-Za-z0-9_$]*|\d+|\S", text.lower())

def build_prompt_query(example):
    nonempty = [line for line in example["left_context"].splitlines() if line.strip()]
    return "\n".join(nonempty[-40:]) + f"\n# current path: {example['file_path']}"

def make_pool_row(example, units, retriever_model, retriever_tokenizer, device):
    query_text = build_prompt_query(example)
    if not units:
        hidden = int(retriever_model.config.hidden_size)
        c = CANDIDATE_POOL_SIZE
        return {
            **{k: example[k] for k in ("task_id", "file_path", "left_context", "target_code", "language")},
            "query_embedding": np.zeros(hidden, dtype=np.float16).tolist(),
            "candidate_embeddings": np.zeros((c, hidden), dtype=np.float16).tolist(),
            "edge_types": np.zeros((c, c), dtype=np.int64).tolist(),
            "candidate_costs": [0] * c,
            "candidate_scores": [-1e9] * c,
            "candidate_texts": [""] * c,
            "candidate_mask": [False] * c,
        }

    rendered = [render_unit(unit) for unit in units]
    query_embedding = encode_retriever_texts(
        retriever_model, retriever_tokenizer, [query_text],
        RETRIEVER_QUERY_LENGTH, True, device
    )[0]
    document_embeddings = encode_retriever_texts(
        retriever_model, retriever_tokenizer, rendered,
        RETRIEVER_CANDIDATE_LENGTH, False, device
    )
    dense_scores = (document_embeddings @ query_embedding).numpy()
    corpus = [lexical_tokens(unit.text) for unit in units]
    query_terms = lexical_tokens(query_text)
    if any(corpus):
        bm25_scores = np.asarray(BM25Okapi(corpus).get_scores(query_terms), dtype=np.float32)
    else:
        bm25_scores = np.zeros(len(units), dtype=np.float32)

    dense_order = np.argsort(-dense_scores)
    bm25_order = np.argsort(-bm25_scores)
    dense_rank = np.empty(len(units), dtype=np.int64)
    bm25_rank = np.empty(len(units), dtype=np.int64)
    dense_rank[dense_order] = np.arange(len(units))
    bm25_rank[bm25_order] = np.arange(len(units))
    rrf = 1.0 / (60 + dense_rank) + 1.0 / (60 + bm25_rank)
    seed_ids = set(dense_order[:DENSE_PROPOSALS].tolist())
    seed_ids.update(bm25_order[:BM25_PROPOSALS].tolist())

    all_edges = typed_relation_matrix(units)
    expansion = set()
    for seed_id in sorted(seed_ids, key=lambda idx: -rrf[idx])[:8]:
        linked = np.flatnonzero(all_edges[seed_id] > 0)
        for linked_id in linked:
            if linked_id != seed_id:
                expansion.add(int(linked_id))
    proposal_ids = seed_ids | expansion
    ordered = sorted(proposal_ids, key=lambda idx: (-rrf[idx], -dense_scores[idx], idx))
    chosen = ordered[:CANDIDATE_POOL_SIZE]

    c = CANDIDATE_POOL_SIZE
    hidden = int(query_embedding.shape[0])
    cand_emb = np.zeros((c, hidden), dtype=np.float16)
    edges = np.zeros((c, c), dtype=np.int64)
    costs = [0] * c
    scores = [-1e9] * c
    texts = [""] * c
    mask = [False] * c
    for slot, original_id in enumerate(chosen):
        unit = units[original_id]
        cand_emb[slot] = document_embeddings[original_id].numpy().astype(np.float16)
        costs[slot] = len(GEN_TOKENIZER.encode(rendered[original_id], add_special_tokens=False)) + 2
        scores[slot] = float(dense_scores[original_id])
        texts[slot] = rendered[original_id]
        mask[slot] = True
        for other_slot, other_id in enumerate(chosen):
            edges[slot, other_slot] = all_edges[original_id, other_id]

    return {
        **{k: example[k] for k in ("task_id", "file_path", "left_context", "target_code", "language")},
        "query_embedding": query_embedding.numpy().astype(np.float16).tolist(),
        "candidate_embeddings": cand_emb.tolist(),
        "edge_types": edges.tolist(),
        "candidate_costs": costs,
        "candidate_scores": scores,
        "candidate_texts": texts,
        "candidate_mask": mask,
    }

def units_for_related_files(example, related_files):
    units = []
    parse_error_files = 0
    for item in related_files[:MAX_RELATED_FILES]:
        if isinstance(item, dict):
            path, source = str(item.get("path", "")), str(item.get("text", ""))
        else:
            path, source = str(item[0]), str(item[1])
        if not path or not source.strip():
            continue
        try:
            file_units, had_error = ast_chunk_file(
                path, source, example["language"], GEN_TOKENIZER,
                AST_CHUNK_TOKENS, max_chunks=MAX_CHUNKS_PER_FILE,
            )
        except Exception as exc:
            print(f"Skip parser failure {path}: {type(exc).__name__}: {exc}")
            continue
        parse_error_files += int(had_error)
        units.extend(file_units)
    deduplicated = {}
    for unit in units:
        key = (unit.file_path, unit.start_byte, unit.end_byte)
        deduplicated[key] = unit
    return list(deduplicated.values()), parse_error_files

def pack_examples(examples, split_name, model, tokenizer, device):
    packed, parse_error_total, chunk_total = [], 0, 0
    for example in tqdm(examples, desc=f"AST + proposal pool ({split_name})"):
        units, parse_errors = units_for_related_files(example, example["related_files"])
        parse_error_total += parse_errors
        chunk_total += len(units)
        row = make_pool_row(example, units, model, tokenizer, device)
        row["candidate_scores"] = row["candidate_scores"]
        packed.append(row)
    print(f"{split_name}: {len(packed)} rows, {chunk_total} AST chunks, "
          f"{parse_error_total} files with parser ERROR nodes")
    return packed

retriever_tokenizer = AutoTokenizer.from_pretrained(
    RETRIEVER_MODEL, cache_dir=str(CACHE_DIR), use_fast=False
)
retriever_model = AutoModel.from_pretrained(
    RETRIEVER_MODEL, cache_dir=str(CACHE_DIR)
).to("cuda:0").eval()

train_rows = pack_examples(
    train_examples, "train", retriever_model, retriever_tokenizer, "cuda:0"
)
valid_rows = pack_examples(
    valid_examples, "valid", retriever_model, retriever_tokenizer, "cuda:0"
)
if not train_rows:
    raise RuntimeError("Candidate-pool builder produced no train rows.")
payload_path = WORK_DIR / "policy_data.pt"
torch.save({"train": train_rows, "valid": valid_rows}, payload_path)
del retriever_model
torch.cuda.empty_cache()
print("Cached fixed pools/features:", payload_path)


In [ ]:
# kaggle-vllm stages a runtime matching Kaggle's preinstalled Torch/CUDA stack.
# It does not replace Kaggle's Torch installation. This profile is validated for T4×2.
sdk_spec = importlib.util.find_spec("kaggle_vllm")
if sdk_spec is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "kaggle-vllm[hub]==0.2.0"
    ])

runtime_root = WORK_DIR / "kaggle-vllm-runtime"
runtime_manifest = runtime_root / "runtime.json"
runtime_root.mkdir(parents=True, exist_ok=True)
if not runtime_manifest.exists():
    bootstrap = [
        "kaggle-vllm", "bootstrap", "--strict",
        "--cache", str(WORK_DIR / "kaggle-vllm-cache"),
        "--staged", str(runtime_root / "vllm-staged"),
        "--overlay", str(runtime_root / "vllm-overlay"),
        "--manifest", str(runtime_manifest),
    ]
    subprocess.run(bootstrap, check=True)
else:
    print("Reuse existing staged vLLM runtime:", runtime_manifest)

server_log_path = WORK_DIR / "vllm-server.log"
server_log = open(server_log_path, "w", encoding="utf-8")
server_env = os.environ.copy()
server_env["CUDA_VISIBLE_DEVICES"] = "0"
server_env["HF_HOME"] = str(CACHE_DIR)
server_env["GENERATOR_MODEL"] = GENERATOR_MODEL
server_env["SERVED_MODEL_NAME"] = SERVED_MODEL_NAME
server_env["VLLM_PORT"] = str(VLLM_PORT)
server_env["VLLM_GPU_MEMORY_UTILIZATION"] = str(VLLM_GPU_MEMORY_UTILIZATION)
server_env["VLLM_MAX_NUM_SEQS"] = str(VLLM_MAX_NUM_SEQS)

serve_shell = (
    'eval "$(kaggle-vllm env)" && '
    'exec vllm serve "$GENERATOR_MODEL" '
    '--host 127.0.0.1 --port "$VLLM_PORT" '
    '--served-model-name "$SERVED_MODEL_NAME" '
    '--dtype float16 --tensor-parallel-size 1 '
    '--max-model-len 2048 '
    '--gpu-memory-utilization "$VLLM_GPU_MEMORY_UTILIZATION" '
    '--max-num-seqs "$VLLM_MAX_NUM_SEQS" --enforce-eager'
)
server_process = subprocess.Popen(
    ["bash", "-lc", serve_shell],
    env=server_env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Started local vLLM server PID:", server_process.pid, "log:", server_log_path)

health_url = API_BASE.rsplit("/v1", 1)[0] + "/health"
ready = False
for attempt in range(180):
    if server_process.poll() is not None:
        server_log.flush()
        tail = server_log_path.read_text(encoding="utf-8", errors="replace")[-6000:]
        raise RuntimeError("vLLM exited during startup. Last log lines:\n" + tail)
    try:
        response = requests.get(health_url, timeout=3)
        if response.ok:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(10)
if not ready:
    tail = server_log_path.read_text(encoding="utf-8", errors="replace")[-6000:]
    raise TimeoutError("vLLM did not become ready. Last log lines:\n" + tail)

# Verify scoring, not only generation: the policy reward depends on prompt token log-probs.
probe_prompt = GEN_TOKENIZER.encode("def add(a, b):\n", add_special_tokens=True)
probe_target = GEN_TOKENIZER.encode("    return a + b\n", add_special_tokens=False)
probe_ids = probe_prompt + probe_target
probe = requests.post(
    API_BASE + "/completions",
    json={
        "model": SERVED_MODEL_NAME,
        "prompt": [probe_ids],
        "max_tokens": 0,
        "temperature": 0.0,
        "prompt_logprobs": 1,
    },
    timeout=300,
)
probe.raise_for_status()
probe_choice = probe.json()["choices"][0]
if not probe_choice.get("prompt_logprobs"):
    raise RuntimeError("vLLM responded but prompt_logprobs is empty; cannot train scalar reward.")
print("vLLM generation + prompt-logprob smoke test passed.")


In [ ]:
TORCHRUN_SCRIPT = r"""
import argparse
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import requests
import torch
import torch.distributed as dist
from torch import nn
from torch.distributions import Categorical
from torch.utils.data import DataLoader, DistributedSampler
from transformers import AutoTokenizer

class SlatePolicy(nn.Module):
    def __init__(self, hidden_size, relation_dim=24, mlp_dim=256):
        super().__init__()
        self.hidden_size = hidden_size
        self.relation_embedding = nn.Embedding(5, relation_dim, padding_idx=0)
        feature_dim = 4 * hidden_size + relation_dim + 2
        self.action_mlp = nn.Sequential(
            nn.Linear(feature_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, 1),
        )
        self.stop_head = nn.Sequential(
            nn.Linear(2 * hidden_size + 1, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        nn.init.zeros_(self.action_mlp[-1].weight)
        nn.init.zeros_(self.action_mlp[-1].bias)
        nn.init.zeros_(self.stop_head[-1].weight)
        nn.init.zeros_(self.stop_head[-1].bias)

    def value(self, query):
        return self.value_head(query).squeeze(-1)

    def action_logits(self, query, candidates, edge_types, selected_mask,
                      costs, valid_mask, remaining, budget):
        count = candidates.shape[0]
        if bool(selected_mask.any()):
            state = candidates[selected_mask].mean(dim=0)
        else:
            state = torch.zeros_like(query)

        rel_ids = edge_types.long().clamp_(0, 4)
        rel_vectors = self.relation_embedding(rel_ids)
        source_mask = selected_mask.float().view(count, 1, 1)
        edge_mask = rel_ids.ne(0).float().unsqueeze(-1)
        denom = (source_mask.squeeze(-1) * edge_mask.squeeze(-1)).sum(0).clamp_min(1.0)
        graph_message = (rel_vectors * source_mask * edge_mask).sum(0) / denom.unsqueeze(-1)

        q_expand = query.unsqueeze(0).expand_as(candidates)
        state_expand = state.unsqueeze(0).expand_as(candidates)
        cost_ratio = costs.float().unsqueeze(-1) / max(float(budget), 1.0)
        remaining_ratio = torch.full_like(cost_ratio, float(remaining) / max(float(budget), 1.0))
        features = torch.cat(
            [q_expand, candidates, q_expand * candidates, state_expand,
             graph_message, cost_ratio, remaining_ratio],
            dim=-1,
        )
        residual = self.action_mlp(features).squeeze(-1)
        dense_prior = (q_expand * candidates).sum(-1) * 10.0
        candidate_logits = dense_prior + residual
        stop_features = torch.cat(
            [query, state, torch.tensor([remaining / max(float(budget), 1.0)],
                                        device=query.device, dtype=query.dtype)]
        )
        stop_logit = self.stop_head(stop_features).reshape(1)
        return torch.cat([candidate_logits, stop_logit], dim=0)

def select_slate(model, row, tokenizer, device, budget, max_steps,
                 max_model_len, crossfile_budget, greedy=False):
    query = torch.tensor(row["query_embedding"], dtype=torch.float32, device=device)
    candidates = torch.tensor(row["candidate_embeddings"], dtype=torch.float32, device=device)
    edges = torch.tensor(row["edge_types"], dtype=torch.long, device=device)
    costs = torch.tensor(row["candidate_costs"], dtype=torch.long, device=device)
    valid = torch.tensor(row["candidate_mask"], dtype=torch.bool, device=device)
    selected_mask = torch.zeros_like(valid)
    selected = []
    log_probability = torch.zeros((), dtype=torch.float32, device=device)
    entropy_sum = torch.zeros((), dtype=torch.float32, device=device)
    remaining = int(budget)

    for _ in range(max_steps):
        logits = model.action_logits(
            query, candidates, edges, selected_mask, costs, valid,
            remaining, budget
        )
        available = valid & (~selected_mask) & (costs <= remaining)
        logits = logits.clone()
        logits[:-1] = logits[:-1].masked_fill(~available, -1.0e9)
        distribution = Categorical(logits=logits)
        action = torch.argmax(logits) if greedy else distribution.sample()
        log_probability = log_probability + distribution.log_prob(action)
        entropy_sum = entropy_sum + distribution.entropy()
        action_id = int(action.item())
        if action_id == len(valid):
            break
        selected.append(action_id)
        selected_mask[action_id] = True
        remaining -= int(costs[action_id].item())
    return selected, log_probability, entropy_sum / max(max_steps, 1)

def compose_prompt(row, selected, tokenizer, max_model_len, crossfile_budget):
    context = "\n\n".join(
        row["candidate_texts"][i] for i in selected
        if i < len(row["candidate_texts"]) and row["candidate_texts"][i]
    )
    context_ids = tokenizer.encode(context, add_special_tokens=False)[:crossfile_budget]
    path_ids = tokenizer.encode(
        f"\n\n# file path: {row['file_path']}\n\n", add_special_tokens=False
    )
    left_ids = tokenizer.encode(row["left_context"], add_special_tokens=False)
    allowed_left = max(max_model_len - len(context_ids) - len(path_ids) - 16, 32)
    prompt_ids = context_ids + path_ids + left_ids[-allowed_left:]
    prompt_ids = prompt_ids[-(max_model_len - 1):]
    return tokenizer.decode(prompt_ids, clean_up_tokenization_spaces=False)

def encode_for_scoring(prompt, target, tokenizer, max_model_len, target_cap):
    target_ids = tokenizer.encode(target, add_special_tokens=False)[:target_cap]
    if not target_ids:
        return None
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=True)
    max_prompt = max(1, max_model_len - len(target_ids))
    if len(prompt_ids) > max_prompt:
        bos = prompt_ids[:1]
        tail_room = max(0, max_prompt - len(bos))
        prompt_ids = bos + prompt_ids[-tail_room:] if tail_room else bos
    sequence = prompt_ids + target_ids
    target_start = len(prompt_ids)
    return sequence, target_start, target_ids

def score_prompts(items, tokenizer, args):
    encoded = []
    for prompt, target in items:
        item = encode_for_scoring(
            prompt, target, tokenizer, args.max_model_len, args.target_cap
        )
        if item is None:
            raise RuntimeError("Empty target continuation encountered.")
        encoded.append(item)
    payload = {
        "model": args.served_model,
        "prompt": [item[0] for item in encoded],
        "max_tokens": 0,
        "temperature": 0.0,
        "prompt_logprobs": 1,
    }
    response = requests.post(args.api_base + "/completions", json=payload, timeout=600)
    response.raise_for_status()
    choices = sorted(response.json()["choices"], key=lambda x: x.get("index", 0))
    if len(choices) != len(encoded):
        raise RuntimeError(f"vLLM returned {len(choices)} choices for {len(encoded)} prompts.")
    scores = []
    for choice, (_, target_start, target_ids) in zip(choices, encoded):
        prompt_logprobs = choice.get("prompt_logprobs")
        if prompt_logprobs is None:
            raise RuntimeError("vLLM response omitted prompt_logprobs.")
        total = 0.0
        for offset, token_id in enumerate(target_ids):
            position = target_start + offset
            if position >= len(prompt_logprobs) or prompt_logprobs[position] is None:
                raise RuntimeError("Missing token-level prompt log-probability.")
            token_entry = prompt_logprobs[position]
            token_entry = token_entry.get(str(token_id), token_entry.get(token_id))
            if token_entry is None:
                raise RuntimeError("vLLM did not return the input token's own log-probability.")
            total += float(token_entry["logprob"] if isinstance(token_entry, dict) else token_entry)
        scores.append(total / max(len(target_ids), 1))
    return scores

def dense_budget_baseline(row, budget, max_steps):
    order = np.argsort(-np.asarray(row["candidate_scores"], dtype=np.float32))
    selected, remaining = [], int(budget)
    for index in order:
        index = int(index)
        if index >= len(row["candidate_mask"]) or not row["candidate_mask"][index]:
            continue
        cost = int(row["candidate_costs"][index])
        if cost <= remaining:
            selected.append(index)
            remaining -= cost
        if len(selected) >= max_steps:
            break
    return selected

def sync_gradients(model, world_size):
    for parameter in model.parameters():
        if parameter.grad is None:
            parameter.grad = torch.zeros_like(parameter)
        dist.all_reduce(parameter.grad, op=dist.ReduceOp.SUM)
        parameter.grad.div_(world_size)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--retriever-model", required=True)
    parser.add_argument("--generator-model", required=True)
    parser.add_argument("--served-model", required=True)
    parser.add_argument("--api-base", required=True)
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--budget", type=int, default=384)
    parser.add_argument("--max-steps", type=int, default=3)
    parser.add_argument("--max-model-len", type=int, default=2048)
    parser.add_argument("--target-cap", type=int, default=96)
    parser.add_argument("--crossfile-budget", type=int, default=384)
    parser.add_argument("--local-batch-size", type=int, default=2)
    parser.add_argument("--seed", type=int, default=17)
    args = parser.parse_args()

    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    torch.set_num_threads(2)

    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    model = None
    data = torch.load(args.data, map_location="cpu", weights_only=False)
    train_rows = data["train"]
    valid_rows = data.get("valid", [])
    hidden_size = len(train_rows[0]["query_embedding"])
    model = SlatePolicy(hidden_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2.0e-4, weight_decay=1.0e-4)
    tokenizer = AutoTokenizer.from_pretrained(
        args.generator_model, use_fast=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    sampler = DistributedSampler(
        list(range(len(train_rows))), num_replicas=world_size, rank=rank,
        shuffle=True, seed=args.seed, drop_last=False
    )
    cache_baseline = {}
    best_val_reward = -float("inf")
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if rank == 0:
        print(f"torchrun policy training: {len(train_rows)} examples, {world_size} ranks, "
              f"hidden={hidden_size}, epochs={args.epochs}", flush=True)

    for epoch in range(args.epochs):
        sampler.set_epoch(epoch)
        loader = DataLoader(
            list(range(len(train_rows))), batch_size=args.local_batch_size,
            sampler=sampler, drop_last=False
        )
        epoch_reward = 0.0
        epoch_count = 0

        for batch_indices in loader:
            batch_rows = [train_rows[int(i)] for i in batch_indices.tolist()]
            rollouts = []
            value_predictions = []
            q_batch = []
            for row in batch_rows:
                q = torch.tensor(row["query_embedding"], dtype=torch.float32, device=device)
                q_batch.append(q)
                with torch.enable_grad():
                    value_predictions.append(model.value(q))
                    selected, logp, entropy = select_slate(
                        model, row, tokenizer, device, args.budget, args.max_steps,
                        args.max_model_len, args.crossfile_budget, greedy=False
                    )
                rollouts.append((row, selected, logp, entropy))

            score_items = []
            score_keys = []
            for row, selected, _, _ in rollouts:
                task_id = row["task_id"]
                if task_id not in cache_baseline:
                    score_items.append((compose_prompt(row, [], tokenizer,
                                                       args.max_model_len, args.crossfile_budget),
                                        row["target_code"]))
                    score_keys.append(("baseline", task_id))
                score_items.append((compose_prompt(row, selected, tokenizer,
                                                   args.max_model_len, args.crossfile_budget),
                                    row["target_code"]))
                score_keys.append(("selected", task_id))
            scores = score_prompts(score_items, tokenizer, args) if score_items else []
            batch_baseline = {}
            batch_selected = {}
            for key, value in zip(score_keys, scores):
                kind, task_id = key
                if kind == "baseline":
                    batch_baseline[task_id] = value
                    cache_baseline[task_id] = value
                else:
                    batch_selected[task_id] = value

            rewards = []
            logps = []
            entropies = []
            for row, selected, logp, entropy in rollouts:
                base = cache_baseline[row["task_id"]]
                reward = float(batch_selected[row["task_id"]] - base)
                rewards.append(reward)
                logps.append(logp)
                entropies.append(entropy)
            reward_tensor = torch.tensor(rewards, dtype=torch.float32, device=device)
            value_tensor = torch.stack(value_predictions)
            logp_tensor = torch.stack(logps)
            entropy_tensor = torch.stack(entropies)
            advantage = reward_tensor - value_tensor.detach()
            policy_loss = -(advantage * logp_tensor).mean()
            value_loss = torch.nn.functional.mse_loss(value_tensor, reward_tensor)
            entropy_bonus = entropy_tensor.mean()
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            sync_gradients(model, world_size)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_reward += float(reward_tensor.detach().sum().item())
            epoch_count += len(batch_rows)

        summary = torch.tensor([epoch_reward, epoch_count], dtype=torch.float32, device=device)
        dist.all_reduce(summary, op=dist.ReduceOp.SUM)
        mean_train_reward = float(summary[0].item() / max(summary[1].item(), 1))

        valid_reward_sum, valid_count = 0.0, 0
        if valid_rows:
            valid_sampler = DistributedSampler(
                list(range(len(valid_rows))), num_replicas=world_size, rank=rank,
                shuffle=False, drop_last=False
            )
            valid_loader = DataLoader(
                list(range(len(valid_rows))), batch_size=args.local_batch_size,
                sampler=valid_sampler, drop_last=False
            )
            with torch.no_grad():
                for valid_indices in valid_loader:
                    valid_batch = [valid_rows[int(i)] for i in valid_indices.tolist()]
                    score_items = []
                    for row in valid_batch:
                        selected, _, _ = select_slate(
                            model, row, tokenizer, device, args.budget, args.max_steps,
                            args.max_model_len, args.crossfile_budget, greedy=True
                        )
                        score_items.append((compose_prompt(
                            row, [], tokenizer, args.max_model_len, args.crossfile_budget
                        ), row["target_code"]))
                        score_items.append((compose_prompt(
                            row, selected, tokenizer, args.max_model_len, args.crossfile_budget
                        ), row["target_code"]))
                    val_scores = score_prompts(score_items, tokenizer, args)
                    for offset in range(0, len(val_scores), 2):
                        valid_reward_sum += val_scores[offset + 1] - val_scores[offset]
                        valid_count += 1
        val_summary = torch.tensor(
            [valid_reward_sum, valid_count], dtype=torch.float32, device=device
        )
        dist.all_reduce(val_summary, op=dist.ReduceOp.SUM)
        mean_val_reward = (
            float(val_summary[0].item() / max(val_summary[1].item(), 1))
            if valid_rows else mean_train_reward
        )
        if rank == 0:
            if mean_val_reward > best_val_reward:
                best_val_reward = mean_val_reward
                torch.save(
                    {"model_state": model.state_dict(), "epoch": epoch + 1,
                     "hidden_size": hidden_size,
                     "mean_train_reward": mean_train_reward,
                     "mean_valid_reward": mean_val_reward},
                    output_path,
                )
            print(f"epoch={epoch + 1} train_reward={mean_train_reward:.6f} "
                  f"valid_reward={mean_val_reward:.6f} best={best_val_reward:.6f} "
                  f"checkpoint={output_path}", flush=True)
        dist.barrier()

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
"""


In [ ]:
train_script_path = WORK_DIR / "ast_gr_slate_train.py"
compile(TORCHRUN_SCRIPT, str(train_script_path), "exec")
train_script_path.write_text(TORCHRUN_SCRIPT, encoding="utf-8")
checkpoint_path = WORK_DIR / "slate_policy.pt"

torchrun = shutil.which("torchrun")
if torchrun is None:
    raise RuntimeError("torchrun is missing from the Kaggle PyTorch image.")

command = [
    torchrun, "--standalone", "--nnodes=1", f"--nproc_per_node={NPROC}",
    str(train_script_path),
    "--data", str(payload_path),
    "--output", str(checkpoint_path),
    "--retriever-model", RETRIEVER_MODEL,
    "--generator-model", GENERATOR_MODEL,
    "--served-model", SERVED_MODEL_NAME,
    "--api-base", API_BASE,
    "--epochs", str(EPOCHS),
    "--budget", str(CROSSFILE_TOKEN_BUDGET),
    "--max-steps", str(MAX_SLATE_STEPS),
    "--max-model-len", str(GENERATOR_MAX_MODEL_LEN),
    "--target-cap", str(GENERATOR_TARGET_TOKENS),
    "--crossfile-budget", str(CROSSFILE_TOKEN_BUDGET),
    "--seed", str(SEED),
]
print("Launching:", " ".join(command))
train_env = os.environ.copy()
train_env["TOKENIZERS_PARALLELISM"] = "false"
train_env["OMP_NUM_THREADS"] = "2"
subprocess.run(command, check=True, env=train_env)
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"torchrun ended without a checkpoint: {checkpoint_path}")

# Load the same policy implementation from the exact script used by torchrun.
import importlib.util
spec = importlib.util.spec_from_file_location("ast_gr_slate_train", train_script_path)
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)
policy_checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
policy_device = torch.device("cuda:1")
slate_policy = train_module.SlatePolicy(policy_checkpoint["hidden_size"]).to(policy_device)
slate_policy.load_state_dict(policy_checkpoint["model_state"])
slate_policy.eval()
print("Loaded trained policy epoch:", policy_checkpoint["epoch"],
      "mean reward:", policy_checkpoint["mean_train_reward"])


In [ ]:
import difflib
import importlib.util

def normalize_code(text):
    return "\n".join(line.rstrip() for line in str(text or "").strip().splitlines())

def edit_similarity(prediction, reference):
    return difflib.SequenceMatcher(
        None, normalize_code(prediction), normalize_code(reference)
    ).ratio()

def decode_crossfile_context(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            return []
    if isinstance(value, dict):
        value = [value]
    return [item for item in value if isinstance(item, dict)]

def benchmark_file_candidates(data_root):
    found = []
    roots = [data_root, data_root / "data"]
    if DATASET_INPUT_HINT.exists():
        mounts = [p for p in DATASET_INPUT_HINT.iterdir() if p.is_dir()]
        roots.extend(mounts)
        roots.extend(p / "data" for p in mounts)
    for root in roots:
        candidates = [
            ("cceval_python", root / "cceval/python/test.parquet", "python"),
            ("cceval_java", root / "cceval/java/test.parquet", "java"),
            ("repoeval_line", root / "repoeval/line_level/test_0.parquet", "python"),
            ("repoeval_line", root / "repoeval/line_level/test_1.parquet", "python"),
        ]
        for name, path, language in candidates:
            if path.is_file() and all(existing[1] != path for existing in found):
                found.append((name, path, language))
    return found

def load_benchmark_examples(path, language, name):
    frame = pd.read_parquet(path)
    required = {"path", "left_context", "crossfile_context", "groundtruth"}
    missing = required - set(frame.columns)
    if missing:
        print(f"Skip {path}; missing columns {sorted(missing)}")
        return []
    if EVAL_LIMIT > 0:
        frame = frame.head(EVAL_LIMIT)
    examples = []
    for index, row in frame.iterrows():
        related_files = decode_crossfile_context(row["crossfile_context"])
        left_context = row["left_context"]
        groundtruth = row["groundtruth"]
        if left_context is None or pd.isna(left_context):
            left_context = ""
        if groundtruth is None or pd.isna(groundtruth):
            groundtruth = ""
        example = {
            "task_id": str(row.get("task_id", f"{name}-{index}")),
            "file_path": str(row["path"]),
            "left_context": str(left_context),
            "target_code": str(groundtruth),
            "language": language,
            "related_files": related_files,
        }
        examples.append(example)
    return examples

def dense_budget_selection(row):
    return train_module.dense_budget_baseline(
        row, CROSSFILE_TOKEN_BUDGET, MAX_SLATE_STEPS
    )

def generate_completions(prompts):
    if not prompts:
        return []
    response = requests.post(
        API_BASE + "/completions",
        json={
            "model": SERVED_MODEL_NAME,
            "prompt": prompts,
            "temperature": 0.0,
            "top_p": 1.0,
            "max_tokens": GENERATOR_OUTPUT_TOKENS,
        },
        timeout=900,
    )
    response.raise_for_status()
    choices = sorted(response.json()["choices"], key=lambda x: x.get("index", 0))
    return [choice.get("text", "") for choice in choices]

def evaluate_one_benchmark(name, language, examples):
    if not examples:
        return []
    encoder = AutoModel.from_pretrained(
        RETRIEVER_MODEL, cache_dir=str(CACHE_DIR)
    ).to("cuda:1").eval()
    tokenizer = AutoTokenizer.from_pretrained(
        RETRIEVER_MODEL, cache_dir=str(CACHE_DIR), use_fast=False
    )
    rows = []
    for example in tqdm(examples, desc=f"Prepare {name}"):
        units, _ = units_for_related_files(example, example["related_files"])
        rows.append(make_pool_row(example, units, encoder, tokenizer, "cuda:1"))
    del encoder
    torch.cuda.empty_cache()

    prompts = []
    metadata = []
    for example, row in zip(examples, rows):
        base_selected = dense_budget_selection(row)
        policy_selected, _, _ = train_module.select_slate(
            slate_policy, row, GEN_TOKENIZER, policy_device,
            CROSSFILE_TOKEN_BUDGET, MAX_SLATE_STEPS,
            GENERATOR_MAX_MODEL_LEN, CROSSFILE_TOKEN_BUDGET, greedy=True,
        )
        for method, selected in (
            ("no_retrieval", []),
            ("dense_top_budget_same_pool", base_selected),
            ("ast_gr_slate_policy", policy_selected),
        ):
            prompts.append(train_module.compose_prompt(
                row, selected, GEN_TOKENIZER,
                GENERATOR_MAX_MODEL_LEN, CROSSFILE_TOKEN_BUDGET
            ))
            metadata.append((example, method, selected))

    predictions = generate_completions(prompts)
    output_rows = []
    for (example, method, selected), prediction in zip(metadata, predictions):
        reference = example["target_code"]
        output_rows.append({
            "dataset": name,
            "task_id": example["task_id"],
            "method": method,
            "prediction": prediction,
            "reference": reference,
            "exact_match": int(normalize_code(prediction) == normalize_code(reference)),
            "edit_similarity": edit_similarity(prediction, reference),
            "selected_chunks": selected,
        })
    return output_rows

all_eval_rows = []
for benchmark_name, parquet_path, language in benchmark_file_candidates(DATA_ROOT):
    examples = load_benchmark_examples(parquet_path, language, benchmark_name)
    if examples:
        print(f"Evaluate {benchmark_name}: {len(examples)} examples from {parquet_path}")
        all_eval_rows.extend(evaluate_one_benchmark(benchmark_name, language, examples))

if all_eval_rows:
    prediction_path = WORK_DIR / "benchmark_predictions.jsonl"
    with prediction_path.open("w", encoding="utf-8") as handle:
        for row in all_eval_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    summary = (
        pd.DataFrame(all_eval_rows)
        .groupby(["dataset", "method"])
        .agg(n=("task_id", "count"),
             exact_match=("exact_match", "mean"),
             edit_similarity=("edit_similarity", "mean"))
        .reset_index()
    )
    summary_path = WORK_DIR / "benchmark_summary.csv"
    summary.to_csv(summary_path, index=False)
    print(summary.to_string(index=False))
    print("Predictions:", prediction_path)
    print("Summary:", summary_path)
else:
    print("Không tìm thấy benchmark parquet phù hợp. Training smoke test vẫn đã chạy; "
          "gắn CrossCodeEval/RepoEval để có end-to-end metrics.")

if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server_process.kill()
print("Artifacts saved under:", WORK_DIR)
